# Code 11.7.1: Shaped rewards, GAE, and the PPO losses

The core computations of PPO for language models, all operating on tensors of shape (batch, response length) with a response mask. `shaped_rewards` implements the per-token reward of this section, `gae` computes whitened advantages and value targets with $\gamma = 1$, and `ppo_loss` returns the clipped policy loss, the clipped value loss, and two diagnostics.


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

def masked_mean(x, mask):
    return (x * mask).sum() / mask.sum()

def masked_whiten(x, mask):
    mean = masked_mean(x, mask)
    var = masked_mean((x - mean) ** 2, mask)
    return (x - mean) * torch.rsqrt(var + 1e-8)

class ValueModel(nn.Module):
    """Critic: a transformer with a scalar head at every position."""
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        self.head = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, seqs, attn, P):
        pos = (attn.cumsum(dim=1) - 1).clamp(min=0)
        h = self.backbone(input_ids=seqs, attention_mask=attn, position_ids=pos).last_hidden_state
        return self.head(h[:, P - 1:-1]).squeeze(-1)         # V(s_t) at the position that predicts y_t

def shaped_rewards(logp_old, ref_logp, score, mask, beta):
    """r_t = -beta * log(pi/pi_ref) at every token, plus the RM score at the last token."""
    log_ratio = (logp_old - ref_logp) * mask
    rewards = -beta * log_ratio
    last = mask.sum(dim=1).long() - 1                        # final response token (the EOS, if any)
    rewards[torch.arange(len(score)), last] += score
    return rewards, log_ratio

def gae(rewards, values, mask, gamma=1.0, lam=0.95):
    """Per-token advantages (whitened) and returns; masked positions contribute nothing."""
    B, T = rewards.shape
    adv = torch.zeros_like(rewards)
    last_gae = torch.zeros(B)
    for t in reversed(range(T)):
        cont = mask[:, t + 1] if t + 1 < T else torch.zeros(B)    # 0 after the last real token
        next_value = values[:, t + 1] * cont if t + 1 < T else torch.zeros(B)
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        last_gae = delta + gamma * lam * cont * last_gae
        adv[:, t] = last_gae
    adv = adv * mask
    returns = adv + values
    return masked_whiten(adv, mask), returns

def ppo_loss(logp, logp_old, adv, values, values_old, returns, mask, clip=0.2, vclip=0.2):
    ratio = torch.exp(logp - logp_old)
    pg = torch.max(-adv * ratio, -adv * ratio.clamp(1 - clip, 1 + clip))
    pg_loss = masked_mean(pg, mask)
    v_clipped = values_old + (values - values_old).clamp(-vclip, vclip)
    vf = 0.5 * torch.max((values - returns) ** 2, (v_clipped - returns) ** 2)
    vf_loss = masked_mean(vf, mask)
    with torch.no_grad():
        stats = {"approx_kl": masked_mean(0.5 * (logp - logp_old) ** 2, mask).item(),
                 "clipfrac": masked_mean(((ratio - 1).abs() > clip).float(), mask).item()}
    return pg_loss, vf_loss, stats
